# bootridge on the Rohwer Lo-SES dataset

Runs Andrew Penn's `bootridge` (empirical-Bayes, .632-bootstrap-tuned ridge
regression) on the **Low-SES** subgroup (n = 37) of the Rohwer dataset,
predicting SAT, PPVT and Raven from the five paired-associate (PA) task
scores (n, s, ns, na, ss).

Records, per predictor per outcome: the bootstrap-optimised ridge constant
(lambda), the global ridge prior contribution to posterior precision, and
BF10 (via lnBF10, the numerically stable scale).

In [ ]:
pkg load statistics-resampling

% Load the full Rohwer dataset and subset to the Low-SES group
fid = fopen ('../data/rohwer_data.csv', 'r');
dat = textscan (fid, '%f %q %f %f %f %f %f %f %f %f', ...
                'Delimiter', ',', 'HeaderLines', 1);
fclose (fid);
[group, SES, SAT, PPVT, Raven, n, s, ns, na, ss] = dat{:};

lo = strcmp (SES, 'Lo');
SAT = SAT(lo); PPVT = PPVT(lo); Raven = Raven(lo);
n = n(lo); s = s(lo); ns = ns(lo); na = na(lo); ss = ss(lo);

printf ('Lo-SES sample size: %d\n', sum (lo));


In [ ]:
% Build the design matrix (intercept + 5 continuous PA predictors,
% no SES term since we've already restricted to a single level)
MAT = bootlm (SAT, {n, s, ns, na, ss}, 'model', 'linear', 'nboot', 0, ...
              'display', 'off', 'continuous', [1:5], 'contrasts', 'simple');

Y = [SAT, PPVT, Raven];
X = MAT.X;
predictor_names = {'Intercept', 'n', 's', 'ns', 'na', 'ss'};
outcome_names   = {'SAT', 'PPVT', 'Raven'};


In [ ]:
% Run bootridge. NOTE: this also prints the full posterior summary to
% stdout below, including the "Global ridge prior contribution to posterior
% precision" line -- that figure isn't a separate struct field, so read it
% off the printed output (or capture stdout with evalc() if you need it
% programmatically).
seed  = 1;
nboot = 200;
alpha = 0.05;

S = bootridge (Y, X, [], nboot, alpha, [], 1, seed);

printf ('\nBootstrap-optimised lambda: %.5f\n', S.lambda);
printf ('Predicted R-squared: %.3f\n', S.RSQ_pred);


In [ ]:
% Tidy per-predictor-per-outcome table: coefficient, lnBF10, BF10, lambda
p = numel (predictor_names);
q = numel (outcome_names);

fid = fopen ('../output/bootridge_lo_ses_results.csv', 'w');
fprintf (fid, 'outcome,predictor,coefficient,lnBF10,BF10,lambda\n');
for j = 1:q
  for i = 1:p
    fprintf (fid, '%s,%s,%.6f,%.6f,%.6f,%.6f\n', outcome_names{j}, ...
              predictor_names{i}, S.coefficient(i, j), S.lnBF10(i, j), ...
              S.BF10(i, j), S.lambda);
  end
end
fclose (fid);
disp ('Saved output/bootridge_lo_ses_results.csv')


**To capture the "Global ridge prior contribution" percentage programmatically**
(rather than reading it from the printed summary above), wrap the call:
```octave
[txt, S] = evalc ('bootridge (Y, X, [], nboot, alpha, [], 1, seed);');
pct = regexp (txt, 'posterior precision: ([\d.]+) %', 'tokens'){1}{1};
```